In [ ]:
import yfinance as yf
import pandas as pd
from typing import Iterable, Union

TARGET_COLS = [
    "underlying_symbol","quote_date","root","expiration","strike","option_type",
    "open","high","low","close","trade_volume",
    "bid_size_1545","bid_1545","ask_size_1545","ask_1545",
    "underlying_bid_1545","underlying_ask_1545",
    "implied_underlying_price_1545","active_underlying_price_1545",
    "implied_volatility_1545","delta_1545","gamma_1545","theta_1545","vega_1545","rho_1545",
    "bid_size_eod","bid_eod","ask_size_eod","ask_eod",
    "underlying_bid_eod","underlying_ask_eod",
    "vwap","open_interest","delivery_code",
]

def _get_underlying_quote(t: yf.Ticker) -> dict:
    last = pd.NA
    try:
        fi = getattr(t, "fast_info", None)
        if isinstance(fi, dict):
            last = fi.get("last_price", pd.NA)
    except Exception:
        pass

    bid = ask = pd.NA
    try:
        info = t.info or {}
        bid = info.get("bid", pd.NA)
        ask = info.get("ask", pd.NA)
    except Exception:
        pass

    return {"last": last, "bid": bid, "ask": ask}

def _to_schema(
    df: pd.DataFrame,
    *,
    underlying_symbol: str,
    expiration: str,
    option_type: str,  # "C" or "P"
    quote_date: pd.Timestamp,
    underlying_quote: dict,
) -> pd.DataFrame:
    out = pd.DataFrame({
        "underlying_symbol": underlying_symbol,
        "quote_date": quote_date.date(),
        "root": underlying_symbol,
        "expiration": pd.to_datetime(expiration).date(),
        "strike": df["strike"].astype(float),
        "option_type": option_type,

        "open": pd.NA,
        "high": pd.NA,
        "low": pd.NA,
        "close": df.get("lastPrice", pd.NA),

        "trade_volume": df.get("volume", pd.NA),

        "bid_size_1545": pd.NA,
        "bid_1545": pd.NA,
        "ask_size_1545": pd.NA,
        "ask_1545": pd.NA,

        "underlying_bid_1545": pd.NA,
        "underlying_ask_1545": pd.NA,

        "implied_underlying_price_1545": pd.NA,
        "active_underlying_price_1545": underlying_quote.get("last", pd.NA),

        "implied_volatility_1545": df.get("impliedVolatility", pd.NA),

        "delta_1545": pd.NA,
        "gamma_1545": pd.NA,
        "theta_1545": pd.NA,
        "vega_1545": pd.NA,
        "rho_1545": pd.NA,

        "bid_size_eod": pd.NA,
        "bid_eod": df.get("bid", pd.NA),
        "ask_size_eod": pd.NA,
        "ask_eod": df.get("ask", pd.NA),

        "underlying_bid_eod": underlying_quote.get("bid", pd.NA),
        "underlying_ask_eod": underlying_quote.get("ask", pd.NA),

        "vwap": pd.NA,
        "open_interest": df.get("openInterest", pd.NA),
        "delivery_code": "",
    })

    # Copy EOD -> 1545 slots
    out["bid_size_1545"] = out["bid_size_eod"]
    out["bid_1545"]      = out["bid_eod"]
    out["ask_size_1545"] = out["ask_size_eod"]
    out["ask_1545"]      = out["ask_eod"]
    out["underlying_bid_1545"] = out["underlying_bid_eod"]
    out["underlying_ask_1545"] = out["underlying_ask_eod"]

    return out[TARGET_COLS]

def fetch_all_expirations_for_symbol(symbol: str) -> pd.DataFrame:
    t = yf.Ticker(symbol)
    expirations = t.options
    if not expirations:
        return pd.DataFrame(columns=TARGET_COLS)

    quote_date = pd.Timestamp.now(tz="America/New_York")
    underlying_quote = _get_underlying_quote(t)

    frames: list[pd.DataFrame] = []
    for exp in expirations:
        try:
            chain = t.option_chain(exp)

            calls = _to_schema(
                chain.calls,
                underlying_symbol=symbol,
                expiration=exp,
                option_type="C",
                quote_date=quote_date,
                underlying_quote=underlying_quote,
            )
            puts = _to_schema(
                chain.puts,
                underlying_symbol=symbol,
                expiration=exp,
                option_type="P",
                quote_date=quote_date,
                underlying_quote=underlying_quote,
            )

            frames.append(calls)
            frames.append(puts)
        except Exception:
            # Skip this expiration if yfinance errors out for it
            continue

    if not frames:
        return pd.DataFrame(columns=TARGET_COLS)

    return pd.concat(frames, ignore_index=True)

def fetch_all_symbols_all_expirations(symbols: Union[str, Iterable[str]]) -> pd.DataFrame:
    if isinstance(symbols, str):
        symbols = [symbols]
    else:
        symbols = list(symbols)

    all_frames: list[pd.DataFrame] = []
    for sym in symbols:
        try:
            all_frames.append(fetch_all_expirations_for_symbol(sym))
        except Exception:
            # Skip the symbol if yfinance errors out for it
            continue

    if not all_frames:
        return pd.DataFrame(columns=TARGET_COLS)

    return pd.concat(all_frames, ignore_index=True)

# Example:



Retrieving options data for expiration date: 2026-01-16

Call Options Data (first 5 rows):
        contractSymbol             lastTradeDate  strike  lastPrice     bid  \
0  AAPL260116C00005000 2026-01-08 20:25:15+00:00     5.0     253.12  252.55   
1  AAPL260116C00010000 2026-01-08 20:25:15+00:00    10.0     247.98  247.55   
2  AAPL260116C00015000 2026-01-09 16:04:55+00:00    15.0     243.06  242.55   
3  AAPL260116C00020000 2026-01-09 16:04:55+00:00    20.0     238.10  238.25   
4  AAPL260116C00025000 2025-12-22 18:27:04+00:00    25.0     245.99  232.60   

      ask    change  percentChange  volume  openInterest  impliedVolatility  \
0  256.15  0.000000       0.000000      16            60          23.679690   
1  251.30  0.000000       0.000000       2            29          10.281254   
2  246.30  0.080002       0.032925       2             2           8.906254   
3  240.95  0.250000       0.105108       2           138           9.375004   
4  236.35  0.000000       0.000000     

In [ ]:
# df = fetch_all_symbols_all_expirations(["NVDA", "LLY", "PLTR", "TSLA", "AMZN", "AAPL"])
df = fetch_all_symbols_all_expirations(["NVDA"])

print(df.head())


,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency
0,AAPL260116C00005000,2026-01-08 20:25:15+00:00,5.0,253.12,252.55,256.15,0.000000,0.000000,16,60,23.679690,True,REGULAR,USD
1,AAPL260116C00010000,2026-01-08 20:25:15+00:00,10.0,247.98,247.55,251.30,0.000000,0.000000,2,29,10.281254,True,REGULAR,USD
2,AAPL260116C00015000,2026-01-09 16:04:55+00:00,15.0,243.06,242.55,246.30,0.080002,0.032925,2,2,8.906254,True,REGULAR,USD
3,AAPL260116C00020000,2026-01-09 16:04:55+00:00,20.0,238.10,238.25,240.95,0.250000,0.105108,2,138,9.375004,True,REGULAR,USD
4,AAPL260116C00025000,2025-12-22 18:27:04+00:00,25.0,245.99,232.60,236.35,0.000000,0.000000,2,15,7.750000,True,REGULAR,USD


In [ ]:

print(df["expiration"].value_counts().head())

Ticker,ValuationTime,Spot,Type,Strike,Expiry,Rate,DividendYield,ContractMultiplier,Bid,Ask,Mid,Last,Vol30d,T,FMV,%Overvalued
str,"datetime[μs, America/New_York]",f64,str,f64,"datetime[μs, America/New_York]",f64,f64,i32,f64,f64,f64,f64,f64,f64,f64,f64
"""TSLA""",2023-08-23 16:00:00 EDT,238.33,"""C""",20.0,2023-08-25 16:00:00 EDT,0.05,0.0,100,218.2,218.35,218.275,218.0,0.486163,5479.452055,230.022572,-0.052267
"""TSLA""",2023-08-23 16:00:00 EDT,238.33,"""P""",20.0,2023-08-25 16:00:00 EDT,0.05,0.0,100,0.0,0.01,0.005,0.01,0.486163,5479.452055,1.385612,-0.992783
"""TSLA""",2023-08-23 16:00:00 EDT,238.33,"""C""",30.0,2023-08-25 16:00:00 EDT,0.05,0.0,100,208.2,208.35,208.275,208.75,0.486163,5479.452055,226.508677,-0.078402
"""TSLA""",2023-08-23 16:00:00 EDT,238.33,"""P""",30.0,2023-08-25 16:00:00 EDT,0.05,0.0,100,0.0,0.01,0.005,0.0,0.486163,5479.452055,2.945357,-1.0
"""TSLA""",2023-08-23 16:00:00 EDT,238.33,"""C""",40.0,2023-08-25 16:00:00 EDT,0.05,0.0,100,198.2,198.35,198.275,198.7,0.486163,5479.452055,223.289066,-0.110122
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""TSLA""",2023-08-23 16:00:00 EDT,238.33,"""P""",540.0,2025-12-19 16:00:00 EST,0.05,0.0,100,297.9,306.0,301.95,0.0,0.486163,2.3261e6,301.67,-1.0
"""TSLA""",2023-08-23 16:00:00 EDT,238.33,"""C""",550.0,2025-12-19 16:00:00 EST,0.05,0.0,100,22.25,22.9,22.575,22.9,0.486163,2.3261e6,238.33,-0.903915
"""TSLA""",2023-08-23 16:00:00 EDT,238.33,"""P""",550.0,2025-12-19 16:00:00 EST,0.05,0.0,100,307.85,315.8,311.825,0.0,0.486163,2.3261e6,311.67,-1.0


In [7]:
df.to_csv('./data/sample_NVDA.csv', index=False)